# display version of python used 

In [1]:
!py --version

Python 3.10.8


# import the libary

In [2]:
from paddleocr import PaddleOCR 
import paddle
import cv2
import numpy as np
import os

D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.3.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `DISABLE_MODEL_SOURCE_CHECK` to `True`.
D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccac

# device used

In [3]:
paddle.device.get_device()

'cpu'

# loading image path

In [4]:
i=1
image_path = f"D:/Anuj_S_Jain/Data/Code/Project/Society_MCE_QT/Society_MCE/Prototype/training data/diff_img/{i}.jpg"
image_path

'D:/Anuj_S_Jain/Data/Code/Project/Society_MCE_QT/Society_MCE/Prototype/training data/diff_img/1.jpg'

# read img

In [5]:
img = cv2.imread(image_path)

# run paddleocr on img 

In [6]:
if (img is None) :
    print(f"{image_path} this is incorrect no img found")
else:
    # setting tread 
    os.environ["OMP_NUM_THREADS"] = "8"
    os.environ["MKL_NUM_THREADS"] = "8"
    # applying ocr on the image
    ocr = PaddleOCR(use_doc_orientation_classify=False, 
        use_doc_unwarping=False, 
        use_textline_orientation=False, 
        lang='en',
        device=paddle.device.get_device(),
        cpu_threads=8 
       )
    result = ocr.predict(img)

Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Anuj S Jain\.paddlex\official_models\PP-OCRv5_server_det`.
Creating model: ('en_PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Anuj S Jain\.paddlex\official_models\en_PP-OCRv5_mobile_rec`.


# make an array by combining the ploy points , text , and confident scores

In [68]:
ocr_result = [(poly , text , scores)
    for poly , text , scores  in 
        zip(result[0]['dt_polys'],
            result[0]['rec_texts'],
            result[0]['rec_scores'])]

# extracting the table from the img 

In [69]:
if (img is None) :
    print(f"{image_path} this is incorrect no img found")
else:
    # converting img to gray
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    # cv2.imwrite(f'./output/opencv/gray{i}.jpeg', gray)

    # splits the image and clahe 
    # C – Contrast
    # L – Limited
    # A – Adaptive
    # H – Histogram
    # E – Equalization
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    gray_clahe = clahe.apply(gray)    
    # cv2.imwrite(f'./output/opencv/gray_clahe{i}.jpeg', gray_clahe)
    
    # Better threshold (adaptive handles uneven lighting)
    thresh = cv2.adaptiveThreshold(
        gray_clahe, 255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY_INV,
        15, 10
    )
    
    
    
    kernel = np.zeros((2,2), np.uint8)
    thresh = cv2.erode(thresh, kernel, iterations=2)
    cv2.imwrite(f'./output/opencv/thresh{i}.jpeg', thresh)
    
    
    # --- Horizontal lines ---
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (50,1))
    horizontal = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horizontal_kernel)
    # cv2.imwrite(f'./output/opencv/horizontal{i}.jpeg', horizontal)
    
    # connect broken horizontal lines
    horizontal = cv2.dilate(horizontal, np.ones((2,2),np.uint8), iterations=2)
    # cv2.imwrite(f'./output/opencv/horizontal_dilated{i}.jpeg', horizontal)
    
    # --- Vertical lines ---
    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1,50))
    vertical = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, vertical_kernel)
    # cv2.imwrite(f'./output/opencv/vertical{i}.jpeg', vertical)
     
    # connect broken vertical lines
    vertical = cv2.dilate(vertical, np.ones((2,2),np.uint8), iterations=2)
    # cv2.imwrite(f'./output/opencv/vertical_dilate{i}.jpeg', vertical)
    
    
    # Combine
    boxes = cv2.add(horizontal, vertical)
    # cv2.imwrite(f'./output/opencv/boxes_combine{i}.jpeg', boxes)
    
    # Final closing to join gaps
    kernel = np.ones((10,10), np.uint8)
    boxes = cv2.morphologyEx(boxes, cv2.MORPH_CLOSE, kernel, iterations=2)
    # cv2.imwrite(f'./output/opencv/boxes_fill_gap{i}.jpeg', boxes)  
    
    # Find contours
    contours, _ = cv2.findContours(boxes, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    # cv2.imwrite(f'./output/opencv/contours{i}.jpeg', contours)
    

# find and draw the counter

In [70]:
output = img.copy()

# for index,cnt in enumerate(contours):
#     x,y,w,h = cv2.boundingRect(cnt)
#     # filter noise
#     if w > 100 and h > 100:
cv2.drawContours(output, contours, -1, (0,255,0), 1)
cv2.imwrite(f'./output/opencv/output{i}.jpeg', output)

True

In [71]:
contours

(array([[[   0,    0]],
 
        ...,
 
        [[2027,    0]]], shape=(82, 1, 2), dtype=int32),
 array([[[ 897, 1420]],
 
        ...,
 
        [[ 838, 1420]]], shape=(66, 1, 2), dtype=int32),
 array([[[1754, 1375]],
 
        ...,
 
        [[1754, 1417]]], shape=(20, 1, 2), dtype=int32),
 array([[[1675, 1374]],
 
        ...,
 
        [[1675, 1416]]], shape=(10, 1, 2), dtype=int32),
 array([[[1495, 1372]],
 
        ...,
 
        [[1495, 1414]]], shape=(16, 1, 2), dtype=int32),
 array([[[ 733, 1372]],
 
        ...,
 
        [[ 629, 1372]]], shape=(27, 1, 2), dtype=int32),
 array([[[1316, 1371]],
 
        ...,
 
        [[1316, 1414]]], shape=(10, 1, 2), dtype=int32),
 array([[[1136, 1371]],
 
        ...,
 
        [[1136, 1413]]], shape=(12, 1, 2), dtype=int32),
 array([[[1017, 1371]],
 
        ...,
 
        [[1017, 1413]]], shape=(10, 1, 2), dtype=int32),
 array([[[ 839, 1371]],
 
        ...,
 
        [[ 838, 1371]]], shape=(14, 1, 2), dtype=int32),
 array([[[1752,  366

# arranging the contours horizontally and vertically 

#### find the center of x and y

In [72]:
def get_center_from_rect(rect):
    x, y, w, h = rect
    cx = x + w / 2
    cy = y + h / 2
    return cx, cy

#### grouping counter vertically 

In [73]:
def group_contours_into_lines_h(contours, x_threshold=20):
    # Step 1: convert contours to bounding boxes 
    rects = [(cv2.boundingRect(cnt),cnt) for cnt in contours]
    #[x1,y1,w1,h1]

    # Step 2: compute centers
    rects_with_centers = [(rect, get_center_from_rect(rect),cnt) for rect,cnt in rects]
    #[[x1,y1,w1,h1],[cx,yc],[org cnt]]

    # Step 3: sort top → bottom
    rects_with_centers.sort(key=lambda r: r[1][0])  # sort by cx

    lines = []
    current_line = []

    for rect, (cx, cy),cnt in rects_with_centers:
        #if the current_line list is emptry append the first element 
        if not current_line:
            current_line.append((rect, cx, cy,cnt))
            continue
            
        #get the last cx
        _, prev_cx,_, _ = current_line[-1]

        # this is the buffer area where all belong to honrisontal 
        if abs(cx - prev_cx) < x_threshold:
            current_line.append((rect, cx, cy,cnt))
        #start as a new line 
        else:
            lines.append(current_line)
            current_line = [(rect, cx, cy,cnt)]

    if current_line:
        lines.append(current_line)

    # Step 4: sort each line left → right
    result = []
    result_org = []
    for line in lines:
        line.sort(key=lambda r: r[2])  # sort by cy
        result.append([rect for rect, _, _,_ in line])
        result_org.append([cnt for _, _, _,cnt in line])

    return result,result_org
    # return lines

#### grouping counter horizontal

In [76]:
def group_contours_into_lines_v(contours, y_threshold=20):
    # Step 1: convert contours to bounding boxes 
    rects = [(cv2.boundingRect(cnt),cnt) for cnt in contours]
    #[x1,y1,w1,h1]

    # Step 2: compute centers
    rects_with_centers = [(rect, get_center_from_rect(rect),cnt) for rect,cnt in rects]
    #[[x1,y1,w1,h1],[cx,yc],[org cnt]]

    # Step 3: sort top → bottom
    rects_with_centers.sort(key=lambda r: r[1][1])  # sort by cy

    lines = []
    current_line = []

    for rect, (cx, cy),cnt in rects_with_centers:
        #if the current_line list is emptry append the first element 
        if not current_line:
            current_line.append((rect, cx, cy,cnt))
            continue
            
        #get the last cy
        _, _, prev_cy,_ = current_line[-1]

        # this is the buffer area where all belong to honrisontal 
        if abs(cy - prev_cy) < y_threshold:
            current_line.append((rect, cx, cy,cnt))
        #start as a new line 
        else:
            lines.append(current_line)
            current_line = [(rect, cx, cy,cnt)]

    if current_line:
        lines.append(current_line)

    # Step 4: sort each line left → right
    result = []
    result_org = []
    for line in lines:
        line.sort(key=lambda r: r[1])  # sort by cy
        result.append([rect for rect, _, _,_ in line])
        result_org.append([cnt for _, _, _,cnt in line])

    return result,result_org
    # return lines

In [77]:
ans_h,ans_org_h = group_contours_into_lines_h(contours)
ans_v,ans_org_v = group_contours_into_lines_v(contours)

In [78]:
len(ans_org)

12

In [79]:
for i_cnt in range(len(ans_org_v)):
    cv2.drawContours(output, ans_org_v[i_cnt], -1, (255,0,0), 3)
    cv2.imwrite(f'./output/opencv/output{i_cnt}.jpeg', output)

# arranging the text box of paddleocr horizontally

In [80]:
def get_center(box):
    #[[x1,y1],[x2,y2],[x3,y3],[x4,y4]]
    x_coords = [p[0] for p in box]
    y_coords = [p[1] for p in box]
    return sum(x_coords) / 4, sum(y_coords) / 4


def group_boxes_into_lines(boxes, y_threshold=20):
    # Step 1: compute centers
    box_centers = [(box, get_center(box),text,score) for box,text,score in boxes]

    # Step 2: sort by Y (top to bottom)
    box_centers.sort(key=lambda b: b[1][1])

    lines = []
    current_line = []

    for box, (cx, cy) ,text ,score in box_centers:
        if score >= 0.9:
            if not current_line:
                current_line.append((box, cx, cy, text, score))
                continue
    
            _, _, prev_cy, _, _ = current_line[-1]
    
            # Step 3: check if same line
            if abs(cy - prev_cy) < y_threshold:
                current_line.append((box, cx, cy, text, score))
            else:
                lines.append(current_line)
                current_line = [(box, cx, cy, text, score)]
        else:
            continue

    if current_line:
        lines.append(current_line)

    # Step 4: sort each line by X (left to right)
    sorted_lines = []
    sorted_lines_text = []
    for line in lines:
        line.sort(key=lambda b: b[2])  # sort by center x
        sorted_lines.append([b[0] for b in line])
        sorted_lines_text.append([b[3] for b in line])

    return sorted_lines,sorted_lines_text

In [81]:
poly,text = group_boxes_into_lines(ocr_result)

In [82]:
poly

[[array([[30,  6],
         ...,
         [27, 46]], shape=(4, 2), dtype=int16),
  array([[299,  12],
         ...,
         [299,  47]], shape=(4, 2), dtype=int16),
  array([[841,  17],
         ...,
         [841,  52]], shape=(4, 2), dtype=int16),
  array([[1028,   19],
         ...,
         [1028,   56]], shape=(4, 2), dtype=int16),
  array([[1357,   21],
         ...,
         [1357,   55]], shape=(4, 2), dtype=int16),
  array([[1534,   19],
         ...,
         [1533,   56]], shape=(4, 2), dtype=int16),
  array([[1811,   21],
         ...,
         [1811,   57]], shape=(4, 2), dtype=int16),
  array([[1148,   18],
         ...,
         [1148,   61]], shape=(4, 2), dtype=int16),
  array([[1673,   23],
         ...,
         [1673,   63]], shape=(4, 2), dtype=int16)],
 [array([[27, 49],
         ...,
         [27, 91]], shape=(4, 2), dtype=int16),
  array([[1028,   60],
         ...,
         [1028,   98]], shape=(4, 2), dtype=int16),
  array([[1311,   60],
         ...,
       

In [83]:
text

[['SI',
  'Description of Goods',
  'HSN/SAC',
  'GST',
  'Rate',
  'Rate',
  'Amount',
  'Quantity',
  'per'],
 ['No.', 'Rate', '(Incl. of Tax)'],
 ['1',
  'Casio FX 991 ES Plus Calculator',
  '84701000',
  '18 %',
  'Pcs',
  '1,225.01',
  '1,038.14',
  '51,907.00',
  '50 Pcs'],
 ['2',
  'Omega Roller Scale 16cms',
  '90172010',
  '18 %',
  '38.00',
  '32.20',
  'Pcs',
  '160 Pcs',
  '5,152.00'],
 ['3',
  'Flair 2mm Lead 5rp',
  '96092000',
  'Pkt',
  '40.00',
  '33.90',
  '169.50',
  '5 Pkt'],
 ['4',
  'Figo Protractor',
  '9017',
  '18 %',
  'Pcs',
  '16.00',
  '13.56',
  '100 Pcs',
  '1,356.00'],
 ['5',
  'VTU Sketch Book',
  '482010',
  '12 %',
  '55.00',
  '49.11',
  'Pcs',
  '9,822.00',
  '200 Pcs'],
 ['68,406.50'],
 ['SGST @ 6%', '6', '589.32'],
 ['CGST @ 6%', '6', '589.32'],
 ['SGST @ 9%', '9', '5,272.61'],
 ['CGST @ 9%', '9', '5,272.61'],
 ['Less :', 'ROUND OFF', '(-)0.36'],
 ['415338'],
 ['Total', '80,130.00'],
 ['Amount Chargeable (in words)'],
 ['INR Eighty Thousand One Hu

In [19]:
ocr_result

[(array([[30,  6],
         ...,
         [27, 46]], shape=(4, 2), dtype=int16),
  'SI',
  0.9859965443611145),
 (array([[299,  12],
         ...,
         [299,  47]], shape=(4, 2), dtype=int16),
  'Description of Goods',
  0.9861448407173157),
 (array([[841,  17],
         ...,
         [841,  52]], shape=(4, 2), dtype=int16),
  'HSN/SAC',
  0.9992126822471619),
 (array([[1028,   19],
         ...,
         [1028,   56]], shape=(4, 2), dtype=int16),
  'GST',
  0.9993073344230652),
 (array([[1148,   18],
         ...,
         [1148,   61]], shape=(4, 2), dtype=int16),
  'Quantity',
  0.9996229410171509),
 (array([[1357,   21],
         ...,
         [1357,   55]], shape=(4, 2), dtype=int16),
  'Rate',
  0.9998060464859009),
 (array([[1534,   19],
         ...,
         [1533,   56]], shape=(4, 2), dtype=int16),
  'Rate',
  0.9994480609893799),
 (array([[1673,   23],
         ...,
         [1673,   63]], shape=(4, 2), dtype=int16),
  'per',
  0.9979965090751648),
 (array([[1811,   21]